# Feature reduction workflow overview

Documented copy describing each stage used to impute missing values, remove redundant variables, and evaluate feature subsets for downstream models.

## Reduction roadmap

Outline the end-to-end plan: clean data, remove redundant features, iterate through model-driven selectors, and retain a final feature shortlist.

## Baseline cleaning strategy

- Impute numeric features via KNN.
- Impute categoricals using modal values.
- Drop columns with more than 10% missingness.

## Environment setup

Import scalers, visualisation libraries, and feature-selection helpers repeatedly used across the reduction experiments.

In [114]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    f1_score,
    recall_score,
    accuracy_score,
    precision_score,
    classification_report,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    fbeta_score,
    make_scorer
)
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
import os
import copy

## Target configuration

Define the classification target and variables marked for removal or renaming before reduction begins.

In [ ]:
TARGET_VARIABLE = "infectes_yes_no"
TARGET_REMOVE = {"infected_yes_no": "infected_yes_no", "sepsis": ["sepsis"], "resultado_hemo": ["all_cult_org", "resultado_hemo"], "all_cult_org": ["all_cult_org", "resultado_hemo"]}
OUTPUT_LOCATION = "/home/pmata/mepram_data/outputs/"
DATABASE_FILE = "/home/pmata/mepram_data/df_merged_full.csv"

## Redundancy filtering

Remove near-zero variance predictors and prune highly correlated features prior to model-based selection.

## Load data and baseline preprocessing

Read the processed dataset, apply KNN and mode imputations, drop columns with excessive missingness, and normalise numeric ranges.

In [116]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer

def evaluate_models_multiclass(models, X_test, y_test, class_names):
    results = []
    classes = np.unique(y_test)
    for model_name, model in models.items():
        y_proba = model.predict_proba(X_test)
        y_pred = np.argmax(y_proba, axis=1)

        metrics = {
            "Model": model_name,
            "F1 (macro)": f1_score(y_test, y_pred, average="macro"),
            "Recall (macro)": recall_score(y_test, y_pred, average="macro"),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision (macro)": precision_score(y_test, y_pred, average="macro"),
            "ROC AUC (macro)": roc_auc_score(y_test, y_proba, labels=model.classes_, multi_class="ovr", average="macro")
        }

        print(f"\nClassification Report for {model_name}:\n")
        print(classification_report(y_test, y_pred, target_names=[str(c) for c in class_names]))
        results.append(metrics)

def safe_drop_columns(df, columns):
    for col in columns:
        try:
            df = df.drop(columns=col)
        except KeyError:
            print(f"Warning: Column {col} not found in DataFrame. Skipping drop.")
    return df

def load_clean_dataset(loaded_df, na_perc_limit, columns_to_delete=[], impute=True):
    print(f"Shape inicial: {loaded_df.shape}")
    missing_cols = [col for col in columns_to_delete if col not in loaded_df.columns]
    if missing_cols:
        print(f"Warning: {len(missing_cols)} columns to delete not found in table: {missing_cols}")
    clean_cols = [col for col in loaded_df.columns if col in columns_to_delete]
    loaded_df = loaded_df.drop(columns=clean_cols, errors="ignore")

    tot = loaded_df.shape[0]

    for col in loaded_df.columns:
        na_per = 1 - len(loaded_df[col].dropna()) / tot
        if na_per > na_perc_limit:
            print(f"Column {col} --> %NaN = {na_per}. deleted")
            loaded_df = loaded_df.drop(columns=col)


    if 'mujer_gestante' in loaded_df.columns:
        loaded_df['mujer_gestante'] = (
            loaded_df['mujer_gestante']
            .map({'False': 0, 'True': 1})  
            .fillna(0)  
        )

    nunique = loaded_df.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()
    loaded_df = loaded_df.drop(columns=constant_cols, errors="ignore")
    print(f"Dropped constant columns: {constant_cols}")

    # Drop date columns
    date_cols = [col for col in loaded_df.columns if "fecha" in col.lower()]
    loaded_df = loaded_df.drop(columns=date_cols, errors="ignore")
    print(f"Dropped date columns: {date_cols}")

    if impute is True:
        df_copy = copy.deepcopy(loaded_df.drop(columns=[TARGET_VARIABLE], errors="ignore"))
        numeric_cols = df_copy.select_dtypes(include=["int", "float"]).columns
        categorical_cols = df_copy.select_dtypes(include=["object", "category"]).columns
        binary_cols = [col for col in numeric_cols if set(df_copy[col].dropna().unique()) <= {0, 1}]

        continuous_cols = [col for col in numeric_cols if col not in binary_cols]
        categorical_numeric_cols = [
            col for col in continuous_cols if df_copy[col].nunique() <15
        ]
        continuous_cols = [col for col in continuous_cols if col not in categorical_numeric_cols]

        if binary_cols:
            binary_imputer = SimpleImputer(strategy="most_frequent")
            df_copy[binary_cols] = binary_imputer.fit_transform(df_copy[binary_cols]).astype(int)

        if continuous_cols:
            numeric_imputer = KNNImputer(n_neighbors=5, weights="distance")
            df_copy[continuous_cols] = numeric_imputer.fit_transform(df_copy[continuous_cols])
            df_copy[continuous_cols] = df_copy[continuous_cols]

        if len(categorical_cols) > 0:
            categorical_imputer = SimpleImputer(strategy="most_frequent")
            df_copy[categorical_cols] = categorical_imputer.fit_transform(df_copy[categorical_cols])
            df_copy[categorical_cols] = df_copy[categorical_cols].astype(str)

        if categorical_numeric_cols:
            categorical_numeric_imputer = SimpleImputer(strategy="most_frequent")
            df_copy[categorical_numeric_cols] = categorical_numeric_imputer.fit_transform(df_copy[categorical_numeric_cols])
            df_copy[categorical_numeric_cols] = df_copy[categorical_numeric_cols].astype(int)
        df_copy[TARGET_VARIABLE] = loaded_df[TARGET_VARIABLE]
        return df_copy

    return loaded_df


def merge_columns(df_to_clean, column_list, new_col_name):
    df_to_clean[new_col_name] = df_to_clean[column_list].sum(axis=1).astype(int)
    clean_df = df_to_clean.drop(columns=column_list)
    return clean_df

df_to_clean = pd.read_csv(DATABASE_FILE)

hepatic_cols = [c for c in df_to_clean.columns if "hepatopatia" in c]
tumor_cols = [c for c in df_to_clean.columns if "cancer" in c]
for new_name, col_list in {"enf_hepaticas": hepatic_cols, "tumores": tumor_cols}.items():
    try:
        df_to_clean = merge_columns(df_to_clean, col_list, new_name)
    except Exception as e:
        print(f"Error merging columns {col_list} into {new_name}: {e}")

columns_to_delete = ["Unnamed: 0", "person_id", "fecha_ingreso_urgencias", "fecha_ingreso_urgencias_x", "shock_septico", "sintoma_nan", "fecha_nacimiento", "codigo_postal", "center", "dag"]
processed_df = load_clean_dataset(
    loaded_df=df_to_clean,
    na_perc_limit=0.052,
    columns_to_delete=columns_to_delete,
    impute=True
)

print("Pre-processing completed. shape without NA's: ", processed_df.dropna().shape)

Shape inicial: (3913, 154)
Column causa_inmunosupresion --> %NaN = 0.7549194991055457. deleted
Column situacion_funcional_basal --> %NaN = 0.10222335803731153. deleted
Column cirugia_previa_con_implant --> %NaN = 1.0. deleted
Column sofa --> %NaN = 0.19626884743163808. deleted
Column snc_glasgow --> %NaN = 0.12701252236135963. deleted
Column bilirrubina --> %NaN = 0.131101456682852. deleted
Column proteina_c_reactiva --> %NaN = 0.1765908510094557. deleted
Column proteina_c_reactiva_recoded --> %NaN = 0.1765908510094557. deleted
Column frec_respiratoria --> %NaN = 0.4221824686940966. deleted
Column saturacion_o2 --> %NaN = 0.05264502938921545. deleted
Column frec_respiratoria_recoded --> %NaN = 0.4221824686940966. deleted
Column saturacion_o2_recoded --> %NaN = 0.05264502938921545. deleted
Dropped constant columns: ['mujer_gestante', 'sintoma_nan_categorico']
Dropped date columns: ['ultima_fecha']
Pre-processing completed. shape without NA's:  (3913, 130)


/tmp/ipykernel_1159995/3789896726.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_copy[TARGET_VARIABLE] = loaded_df[TARGET_VARIABLE]


### Persist baseline dataset

Save the cleaned dataframe used as the starting point for all subsequent reduction strategies.

In [136]:
processed_df.to_csv(os.path.join(OUTPUT_LOCATION, "red_var_df.csv"), index=False)

## Prepare categorical recoding helpers

Rename symptom columns using human-readable labels to keep feature names interpretable after selection.

In [118]:
recode_dictlist = [{'value': '1', 'name': ' fiebre'},
 {'value': '2', 'name': ' tos'},
 {'value': '3', 'name': ' dificultad para respirar'},
 {'value': '4', 'name': ' dolor costal'},
 {'value': '5', 'name': ' disuria'},
 {'value': '6', 'name': ' síndrome de disuria - polaquiuria'},
 {'value': '7', 'name': ' tenesmo de ano y/o recto'},
 {'value': '8', 'name': ' dolor en el ángulo renal'},
 {'value': '9', 'name': ' náuseas'},
 {'value': '10', 'name': ' vómitos'},
 {'value': '11', 'name': ' dolor abdominal'},
 {'value': '12', 'name': ' diarrea'},
 {'value': '13', 'name': ' lesión de la piel'},
 {'value': '14', 'name': ' lesión de mucosa'},
 {'value': '15', 'name': ' cefalea'},
 {'value': '16', 'name': ' dolor articular'},
 ]
recode_dict = {d["value"]: d["name"].strip().replace(" ", "_") for d in recode_dictlist}

In [119]:
try:
    processed_df = processed_df.rename(columns={
        f"sintoma_{key}.0": f"sintoma_{val}" for key, val in recode_dict.items()
    })
    processed_df = processed_df.rename(columns={
        f"sintoma_{key}.0_categorico": f"sintoma_{val}_categorico" for key, val in recode_dict.items()
    })
except Exception as e:
    print(f"Error renaming columns: {e}")

In [120]:
from itertools import combinations
def create_combinations(processed_df, list_features, drop=False):
    for feature1, feature2 in combinations(list_features, 2):
        processed_df[f"{feature1}_{feature2}"] = processed_df[feature1] + processed_df[feature2]
    if drop:
        processed_df.drop(list_features, axis=1)
    return processed_df

In [121]:
processed_df = processed_df[~processed_df["foco"].isin([11.0, 7.0, 10.0, 5.0, 9.0, 8.0])]

### Evaluate post-processing summaries

Inspect dataset shape, class balance, and descriptive stats to confirm preprocessing outcomes.

In [122]:
processed_df.dropna().shape

(3693, 130)

### Helper functions for Bayesian frequency smoothing

Utilities to combine observed event rates with population priors, stabilising feature importance estimates.

### Apply Bayesian priors to frequency counts

Blend population-level priors with observed frequencies before computing feature scores.

In [123]:
from scipy.special import softmax

def compute_priors_by_foco(y_train, foco_train, k_smooth=0.1, exclude_negative=True):
    classes = np.array(sorted(y_train.unique()))
    cnt = pd.crosstab(foco_train, y_train).reindex(columns=classes, fill_value=0)
    if exclude_negative and 2 in cnt.columns:
        positives = [c for c in cnt.columns]# if c != 2]
        cnt_pos = cnt[positives]
        K = len(positives)
        prior_foco = (cnt_pos + k_smooth).div(cnt_pos.sum(axis=1) + k_smooth * K, axis=0)
    else:
        K = len(classes)
        prior_foco = (cnt + k_smooth).div(cnt.sum(axis=1) + k_smooth * K, axis=0)
    #prior_foco[2] = 0
    cnt_global = y_train.value_counts().reindex(classes, fill_value=0)
    prior_global = (cnt_global + k_smooth) / (cnt_global.sum() + k_smooth * len(classes))
    support_foco = cnt.sum(axis=1)
    return classes, prior_foco, prior_global, support_foco

def build_prior_matrix(foco_series, prior_foco, prior_global, support_foco,
                       classes, blend_strength=1):
    """Return matrix (n_samples x n_classes) of blended priors for each row."""
    n, C = len(foco_series), len(classes)
    priors = np.empty((n, C))
    prior_global_arr = prior_global.values.astype(float)
    for i, foco in enumerate(foco_series):
        #priors[i, :] = prior_foco.loc[foco]
        if foco in prior_foco.index:
            p_f = prior_foco.loc[foco].values.astype(float)
            n_f = support_foco.get(foco, 0)
            w = n_f / (n_f + blend_strength)  # backoff to global prior
            priors[i, :] = w * p_f + (1 - w) * prior_global_arr
        else:
            priors[i, :] = prior_global_arr
    return priors

def adjust_proba_logit(y_pred_proba, foco_series, prior_foco, prior_global,
                       support_foco, classes, beta=1.0, blend_strength=1, eps=1e-12):
    """Bayesian logit-space adjustment: log P* = log P_model + beta log P_prior."""
    prior_mat = build_prior_matrix(foco_series, prior_foco, prior_global,
                                   support_foco, classes, blend_strength)
    logits = np.log(np.clip(y_pred_proba, eps, 1.0))
    log_prior = np.log(np.clip(prior_mat, eps, 1.0))
    adj_logits = logits + beta * log_prior
    return softmax(adj_logits, axis=1)

def adjust_with_p_f_given_b(y_pred_proba, foco_series, pf_b, classes,
                            beta=1.0, blend_strength=None, prior_global_f=None, eps=1e-12):
    """
    y_pred_proba: (n_samples x n_classes) from the model
    foco_series: Series of 'foco' for those samples
    pf_b: DataFrame, index=foco, columns=classes, values=P(foco|bacteria)
    classes: array-like class order used by the model
    beta: strength of the prior (>=0)
    blend_strength: if not None, blend P(f|b) toward a global P(f) when bacteria has low support
    prior_global_f: Series P(foco) over TRAIN (needed only if blend_strength is used)
    """
    n, C = y_pred_proba.shape
    assert C == len(classes)

    # logits of the model posteriors
    logits = np.log(np.clip(y_pred_proba, eps, 1.0))

    # Build per-row log-prior term log P(f|b) for the observed foco in each row
    log_prior = np.zeros_like(logits)
    for i, foco in enumerate(foco_series):
        if foco in pf_b.index:
            p_vec = pf_b.loc[foco].reindex(classes).fillna(0.0).to_numpy()
            # Optional blending to a global foco distribution if desired
            if blend_strength is not None and prior_global_f is not None:
                # Blend per-bacteria via support (columns’ counts)
                # (here we use a single foco row, so we need class-wise blending)
                # For simplicity, use same weight for all classes or precompute supports per class.
                # If you tracked support per bacteria column, use it here.
                # Otherwise skip blending or keep a fixed small weight to avoid zeros.
                pass
            log_prior[i, :] = np.log(np.clip(p_vec, eps, 1.0))
        else:
            # Foco unseen: neutral adjustment (no-op)
            log_prior[i, :] = 0.0

    adj_logits = logits + beta * log_prior
    return softmax(adj_logits, axis=1)

In [124]:
def create_bac_foco_freq_matrix(df):
    freq_foco_resultado = (
        df.groupby(["foco", "resultado_hemo"], observed=True)
        .size()
        .unstack(fill_value=0)             # columns = foco, rows = resultado_hemo
    )
    freq_foco_resultado_pct = freq_foco_resultado.div(freq_foco_resultado.sum(axis=0), axis=1)
    return freq_foco_resultado_pct

## Variable reduction for multi-class targets

Tailor feature selection to multiclass prediction tasks, including organism-specific targets.

### Multiclass reduction workflow

Sequence the steps required to tailor feature selection for multiclass prediction targets.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, f1_score, precision_recall_curve, average_precision_score
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import pandas as pd
import numpy as npa
import copy
from itertools import combinations
from collections import defaultdict
import copy

from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.feature_selection import RFECV
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    fbeta_score,
    make_scorer,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import label_binarize, MinMaxScaler
from sklearn.naive_bayes import MultinomialNB

invalid_cols = ["sepsis", "qsofa", "vasopresores", "hipotension", TARGET_VARIABLE, "freq_bacteria", "freq_bac_foco", TARGET_REMOVE[TARGET_VARIABLE]]
invalid_orgs = ["Enterococcus", "_Fungi", "_Other bacteria"]#, "_Enterobacteria"]
invalid_focos = ["catéter venoso", "vías altas respiratorias", "cardiovascular", "osteoarticular", "sistema nervioso central", "genital"]
cols_to_encode = ["ultimo_antib"]
test_df = copy.deepcopy(processed_df)

print("Initial dataset shape: ", test_df.shape)

test_df = test_df[~test_df[TARGET_VARIABLE].isin(invalid_orgs)]
test_df = test_df[~test_df["foco"].isin(invalid_focos)]
test_df = test_df.dropna()

print("Final dataset shape: ", test_df.shape)

X = safe_drop_columns(test_df, invalid_cols)
for col in cols_to_encode:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
le = LabelEncoder()
y = le.fit_transform(test_df[TARGET_VARIABLE])
classes = np.unique(y)
mapping = dict(zip(le.classes_, range(len(le.classes_))))
print("Class mapping:", mapping)

y = pd.Series(y, name=TARGET_VARIABLE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=99
)
classes, prior_foco, prior_global, support_foco = compute_priors_by_foco(
    y_train, X_train["foco"]
)
tbl = (
    X_train.assign(bacteria=y_train)  # y_train are class labels aligned to X_train
          .pivot_table(index="foco", columns="bacteria", values=None, aggfunc="size", fill_value=0)
)

k = 1.0
col_sums = tbl.sum(axis=0)
pf_b = (tbl + k).div(col_sums + k * tbl.shape[0], axis=1)
support_b = col_sums
classes = pf_b.columns.to_numpy()

scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

min_class_samples = y_train.value_counts().min()
smote_k = max(1, min(5, min_class_samples - 1))
counts = y.value_counts()
max_class = counts.idxmax()
max_count = counts.max()
target_frac = 0.30
target_size = int(max_count * target_frac)

sampling_strategy = {}
for cls, count in counts.items():
    if cls == max_class:
        sampling_strategy[cls] = count
    else:
        target = min(count * 10, target_size)
        sampling_strategy[cls] = max(count, target)
sampler = SMOTE(random_state=99, k_neighbors=smote_k, sampling_strategy=sampling_strategy)

X_train_balanced, y_train_balanced = sampler.fit_resample(X_train_scaled, y_train)
X_train_balanced = pd.DataFrame(X_train_balanced, columns=X_train.columns)
y_train_balanced = pd.Series(y_train_balanced, name=TARGET_VARIABLE)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        bootstrap=True,
        random_state=99,
        n_jobs=-1
    ),
    "Logistic Regression": LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        l1_ratio=0.5,
        C=0.3,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
       objective="multiclass",
       num_class=y.nunique(),
       boosting_type="gbdt",
       learning_rate=0.05,
       n_estimators=500,
       max_depth=-1,
       subsample=0.8,
       colsample_bytree=0.8,
       random_state=99,
       n_jobs=-1,
       verbose=-1,
       verbosity=-1,
    ),
    "CatBoost": CatBoostClassifier(
         iterations=300,
         learning_rate=0.05,
         depth=8,
         loss_function="MultiClass",
         auto_class_weights="Balanced",
         verbose=False
    )
    
    #"GaussianBayes": MultinomialNB(
    #    alpha= 0.5,         # smoothing strength (0 → no smoothing, 1 → strong)
    #    fit_prior=True,    # learn class priors from data
    #    force_alpha= True   # ensures alpha is always positive (since sklearn 1.4+)
    #)
}
cv_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
scorer = make_scorer(fbeta_score, beta=2, average="macro")


Initial dataset shape:  (3693, 130)
Final dataset shape:  (3495, 130)
Class mapping: {'Escherichia coli': 0, 'Klebsiella pneumoniae': 1, 'NEGATIVE': 2, 'Pseudomonas aeruginosa': 3, 'Staphylococcus aureus': 4, 'Streptococcus pneumoniae': 5, '_Enterobacteria': 6}


In [126]:
foco_map = {1.0: 'pulmonar',
 2.0: 'intraabdominal',
 3.0: 'biliar',
 4.0: 'urinario',
 5.0: 'cardiovascular',
 6.0: 'piel',
 7.0: 'sistema nervioso central',
 8.0: 'catéter venoso',
 9.0: 'vías altas respiratorias',
 10.0: 'osteoarticular',
 11.0: 'genital',
 12.0: 'desconocido'}
class_mapping = {'Escherichia coli': 0, 'Klebsiella pneumoniae': 1, 'NEGATIVE': 2, 'Pseudomonas aeruginosa': 3, 'Staphylococcus aureus': 4, 'Streptococcus pneumoniae': 5, "_Enterobacteria": 6}


Check priors by "foco"

In [127]:
prior_foco.rename(index=foco_map).rename(columns={v:k for k,v in class_mapping.items()})

all_cult_org,Escherichia coli,Klebsiella pneumoniae,NEGATIVE,Pseudomonas aeruginosa,Staphylococcus aureus,Streptococcus pneumoniae,_Enterobacteria,7
foco,,,,,,,,
pulmonar,0.268731,0.080338,0.441966,0.030533,0.032698,0.037029,0.056518,0.052187
intraabdominal,0.293991,0.057940,0.451359,0.036481,0.043634,0.022175,0.043634,0.050787
biliar,0.262363,0.070055,0.399725,0.056319,0.028846,0.111264,0.028846,0.042582
urinario,0.302012,0.081815,0.402620,0.019172,0.055239,0.032460,0.059036,0.047646
piel,0.266962,0.163717,0.325959,0.045723,0.045723,0.060472,0.075221,0.016224
desconocido,0.232350,0.037408,0.511591,0.011064,0.058483,0.047945,0.058483,0.042677


In [128]:
from sklearn.base import BaseEstimator, ClassifierMixin

class PriorAdjustedClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator, pf_b, classes, beta_prior=1.0):
        self.base_estimator = base_estimator
        self.pf_b = pf_b
        self.classes_ = np.array(classes)
        self.beta_prior = beta_prior

    def fit(self, X, y):
        self.base_estimator.fit(X, y)
        return self

    def predict_proba(self, X):
        y_proba = self.base_estimator.predict_proba(X)
        foco_series = X["foco"]
        return adjust_with_p_f_given_b(
            y_pred_proba=y_proba,
            foco_series=foco_series,
            pf_b=self.pf_b,
            classes=self.classes_,
            beta=self.beta_prior
        )

    def predict(self, X):
        y_proba_adj = self.predict_proba(X)
        return self.classes_[np.argmax(y_proba_adj, axis=1)]

In [129]:
def make_prior_adjusted_scorer(pf_b, classes, beta_prior=1.0):
    def prior_adjusted_f2(estimator, X, y_true):
        # Get model predictions
        y_proba = estimator.predict_proba(X)
        foco_series = X["foco"]  # ensure foco is in X
        # Adjust probabilities with priors
        y_proba_adj = adjust_with_p_f_given_b(
            y_pred_proba=y_proba,
            foco_series=foco_series,
            pf_b=pf_b,
            classes=classes,
            beta=beta_prior
        )
        y_pred = classes[np.argmax(y_proba_adj, axis=1)]
        return fbeta_score(y_true, y_pred, beta=2, average="macro")
    return make_scorer(prior_adjusted_f2, greater_is_better=True)

Check posterior frequencies

In [ ]:
pf_b.rename(index=foco_map).rename(columns={v:k for k,v in class_mapping.items()})

In [ ]:
print([x for x in processed_df.columns])

## RFE approach for compact feature sets

Iteratively shrink the feature space to a minimal subset while monitoring performance metrics.

### Binary RFE experiment loop

Iterate over estimator choices, scoring each subset while decrementing feature counts.

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.metrics import (
    make_scorer, fbeta_score, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve,
    classification_report
)
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.feature_selection import RFE
from sklearn.base import clone
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline as SkPipeline

# --- Scoring function
macro_f2_scorer = make_scorer(fbeta_score, beta=2, average="macro")

# --- Configuration
feature_counts = [40, 20, 15, 10]
results2_all_vars_rfe, results2_rfe = {}, {}
curves2_all_rfe, curves2_rfe = defaultdict(dict), defaultdict(dict)
important_features2_rfe = {}

cv_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
rng = np.random.default_rng(99)

min_class_samples_cv = int(y_train.value_counts().min())
use_smote_in_cv = min_class_samples_cv >= 5
smote_k_cv = max(1, min(3, min_class_samples_cv - 1)) if use_smote_in_cv else None

for name, estimator in models.items():
    print(f"\n=== Modelo: {name} ===")

    # --- Build pipeline (scaler -> SMOTE -> model)
    cv_steps = [("scaler", MinMaxScaler())]
    if use_smote_in_cv:
        cv_steps.append((
            "smote",
            SMOTE(
                random_state=99,
                sampling_strategy="not majority",
                k_neighbors=smote_k_cv,
            ),
        ))
        cv_pipeline = ImbPipeline(cv_steps + [("model", clone(estimator))])
    else:
        cv_pipeline = SkPipeline(cv_steps + [("model", clone(estimator))])

    # --- Cross-validation on raw training data (no pre-balancing!)
    cv_results = cross_validate(
        cv_pipeline,
        X_train, y_train,
        cv=cv_kfold,
        scoring={"f2_macro": macro_f2_scorer},
        n_jobs=-1,
    )

    cv_f2_mean = float(cv_results["test_f2_macro"].mean())
    cv_f2_std = float(cv_results["test_f2_macro"].std())
    print(f"  CV stratified F2-macro={cv_f2_mean:.3f} ± {cv_f2_std:.3f}")

    # --- Permutation sanity check
    y_perm = rng.permutation(y_train.values)
    perm_scores = cross_val_score(
        cv_pipeline,
        X_train, y_perm,
        cv=cv_kfold,
        scoring=macro_f2_scorer,
        n_jobs=-1,
    )
    perm_f2_mean = float(perm_scores.mean())
    perm_f2_std = float(perm_scores.std())
    print(f"  Permuted-label CV F2-macro={perm_f2_mean:.3f} ± {perm_f2_std:.3f}")

    # --- Train model on full training set using same pipeline
    final_pipeline = clone(cv_pipeline)
    final_pipeline.fit(X_train, y_train)

    # Get scaled test data using pipeline's scaler
    # (extract scaler to apply same transform)
    scaler = final_pipeline.named_steps.get("scaler", None)
    if scaler is not None:
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test),
            columns=X_train.columns,
            index=X_test.index,
        )
    else:
        X_test_scaled = X_test.copy()

    base_model = final_pipeline.named_steps["model"]
    classes = base_model.classes_
    y_test_bin = label_binarize(y_test, classes=classes)
    y_proba = base_model.predict_proba(X_test_scaled)
    y_pred = classes[np.argmax(y_proba, axis=1)]

    macro_f2 = fbeta_score(y_test, y_pred, beta=2, average="macro")
    macro_auc = roc_auc_score(y_test_bin, y_proba, multi_class="ovr", average="macro")
    macro_ap = average_precision_score(y_test_bin, y_proba, average="macro")

    results2_all_vars_rfe[name] = {
        "macro_f2": macro_f2,
        "macro_auc": macro_auc,
        "macro_avg_precision": macro_ap,
        "cv_macro_f2_mean": cv_f2_mean,
        "cv_macro_f2_std": cv_f2_std,
        "perm_cv_macro_f2_mean": perm_f2_mean,
        "perm_cv_macro_f2_std": perm_f2_std,
    }

    # Per-class ROC/PR curves
    roc_curves, pr_curves = [], []
    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        prec, rec, _ = precision_recall_curve(y_test_bin[:, i], y_proba[:, i])
        roc_curves.append((fpr.tolist(), tpr.tolist()))
        pr_curves.append((prec.tolist(), rec.tolist()))
    curves2_all_rfe[name]["roc"] = roc_curves
    curves2_all_rfe[name]["pr"] = pr_curves

    # --- Recursive Feature Elimination (RFE)
    best_score = -np.inf
    best_subset, best_model, best_metrics = [], None, {}

    for n_feats in feature_counts:
        if n_feats >= X_train.shape[1]:
            continue

        rfe = RFE(estimator=clone(estimator), n_features_to_select=n_feats, step=1)
        rfe.fit(X_train, y_train)
        subset = X_train.columns[rfe.support_]
        subset_list = list(subset)

        model_sel = clone(estimator)
        model_sel.fit(X_train[subset_list], y_train)

        y_proba_sel = model_sel.predict_proba(X_test_scaled[subset_list])

        # Optional: Adjust probabilities with priors (train-only priors)
        y_proba_sel_adj = adjust_proba_logit(
            y_pred_proba=y_proba_sel,
            foco_series=X_test["foco"],
            prior_foco=prior_foco,
            prior_global=prior_global,
            support_foco=support_foco,
            classes=classes,
            beta=0.9,
            blend_strength=5 
        )

        y_pred_sel = model_sel.classes_[np.argmax(y_proba_sel_adj, axis=1)]
        y_test_bin_sel = label_binarize(y_test, classes=model_sel.classes_)

        macro_f2_sel = fbeta_score(y_test, y_pred_sel, beta=2, average="macro")
        macro_auc_sel = roc_auc_score(y_test_bin_sel, y_proba_sel_adj, multi_class="ovr", average="macro")
        macro_ap_sel = average_precision_score(y_test_bin_sel, y_proba_sel_adj, average="macro")

        if macro_f2_sel >= best_score:
            best_score = macro_f2_sel
            best_subset = subset_list
            best_model = model_sel
            best_metrics = {
                "macro_f2": float(macro_f2_sel),
                "macro_auc": float(macro_auc_sel),
                "macro_avg_precision": float(macro_ap_sel),
                "roc": [(roc_curve(y_test_bin_sel[:, i], y_proba_sel_adj[:, i])[:2]) for i in range(len(classes))],
                "pr": [(precision_recall_curve(y_test_bin_sel[:, i], y_proba_sel_adj[:, i])[:2]) for i in range(len(classes))],
            }

        print(f"  n_features={n_feats:<3d} | F2-macro={macro_f2_sel:.3f} | AUC={macro_auc_sel:.3f} | AP={macro_ap_sel:.3f}")

    important_features2_rfe[name] = list(best_subset)
    results2_rfe[name] = {
        "macro_f2": best_metrics["macro_f2"],
        "macro_auc": best_metrics["macro_auc"],
        "macro_avg_precision": best_metrics["macro_avg_precision"],
        "selected_features": list(best_subset),
    }
    curves2_rfe[name]["roc"] = best_metrics["roc"]
    curves2_rfe[name]["pr"] = best_metrics["pr"]

    print("\nBest subset for", name)
    print(classification_report(y_test, best_model.predict(X_test_scaled[list(best_subset)])))
    print(f"Selected {len(best_subset)} features | F2-macro={best_metrics['macro_f2']:.3f} | AUC={best_metrics['macro_auc']:.3f}")



=== Modelo: Random Forest ===
  CV stratified F2-macro=0.237 ± 0.013
  Permuted-label CV F2-macro=0.126 ± 0.004


/home/pmata/micromamba/envs/data_analysis/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


  n_features=40  | F2-macro=0.124 | AUC=0.663 | AP=0.207
  n_features=20  | F2-macro=0.124 | AUC=0.603 | AP=0.192


In [ ]:
import matplotlib.pyplot as plt

# === ROC Curves ===
plt.figure(figsize=(8, 6))
for name, curve_data in curves2_rfe.items():
    fpr, tpr = curve_data["roc"]
    plt.plot(fpr, tpr, lw=2, label=f"{name}")
plt.plot([0, 1], [0, 1], "k--", lw=1.5, label="Random")
plt.title("ROC Curves (RFE Models)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

# === Precision–Recall Curves ===
plt.figure(figsize=(8, 6))
for name, curve_data in curves2_rfe.items():
    precision, recall = curve_data["pr"]
    plt.plot(recall, precision, lw=2, label=f"{name}")
plt.title("Precision–Recall Curves (RFE Models)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

calibration_curve(y_pred)

def plot_metrics_multiclass(models, X_test, y_test, classes, class_names):
    # ROC curves
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    for model_name, model in models.items():
        y_proba = model.predict_proba(X_test)
        for i, cls in enumerate(classes):
            fpr, tpr, _ = roc_curve((y_test == cls).astype(int), y_proba[:, i])
            plt.plot(fpr, tpr, label=f"{model_name} - class {class_names[i]}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC Curves (per class)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()

    # PR curves
    plt.subplot(1, 2, 2)
    for model_name, model in models.items():
        y_proba = model.predict_proba(X_test)
        for i, cls in enumerate(classes):
            precision, recall, _ = precision_recall_curve((y_test == cls).astype(int), y_proba[:, i])
            plt.plot(recall, precision, label=f"{model_name} - class {class_names[i]}")
    plt.title("Precision-Recall Curves (per class)")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Confusion matrices
    for model_name, model in models.items():
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred, labels=classes)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix for {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.show()

plot_metrics_multiclass(models, X_test, y_test, classes, list(mapping.keys()))

## RFE without Bayesian adjustment (raw bacterial frequency)

Benchmark the impact of omitting Bayesian priors when informing RFE-based selections.

### Compare raw vs Bayesian-informed RFE

Assess how removing Bayesian smoothing alters selected features and model metrics.

In [86]:
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.metrics import make_scorer

macro_f2_scorer = make_scorer(fbeta_score, beta=2, average="macro")

feature_counts = [40, 20, 15, 10]
results_all_vars_rfe, results_rfe = {}, {}
curves_all_rfe, curves_rfe = defaultdict(dict), defaultdict(dict)
important_features_rfe = {}

cv_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
rng = np.random.default_rng(99)
min_class_samples_cv = int(y_train.value_counts().min())
use_smote_in_cv = min_class_samples_cv > 1
smote_k_cv = max(1, min(3, min_class_samples_cv - 1)) if use_smote_in_cv else None

for name, estimator in models.items():
    print(f"Modelo: {name}")

    cv_steps = [("scaler", MinMaxScaler())]
    if use_smote_in_cv:
        cv_steps.append((
            "smote",
            SMOTE(
                random_state=99,
                sampling_strategy="not majority",
                k_neighbors=smote_k_cv,
            ),
        ))
        cv_pipeline = ImbPipeline(cv_steps + [("model", clone(estimator))])
    else:
        cv_pipeline = SkPipeline(cv_steps + [("model", clone(estimator))])

    cv_results = cross_validate(
        cv_pipeline,
        X_train,
        y_train,
        cv=cv_kfold,
        scoring={"f2_macro": macro_f2_scorer},
        n_jobs=-1,
    )
    cv_f2_mean = float(cv_results["test_f2_macro"].mean())
    cv_f2_std = float(cv_results["test_f2_macro"].std())
    print(
        f"  CV stratified F2-macro={cv_f2_mean:.3f} ± {cv_f2_std:.3f} (training-only validation)"
    )

    perm_pipeline = clone(cv_pipeline)
    y_perm = rng.permutation(y_train.values)
    perm_scores = cross_val_score(
        perm_pipeline,
        X_train,
        y_perm,
        cv=cv_kfold,
        scoring=macro_f2_scorer,
        n_jobs=-1,
    )
    perm_f2_mean = float(perm_scores.mean())
    perm_f2_std = float(perm_scores.std())
    print(
        f"  Permuted-label CV F2-macro={perm_f2_mean:.3f} ± {perm_f2_std:.3f} (sanity check)"
    )

    base_model = clone(estimator)
    base_model.fit(X_train_balanced, y_train_balanced)

    classes = base_model.classes_
    y_test_bin = label_binarize(y_test, classes=classes)
    y_proba = base_model.predict_proba(X_test_scaled)
    y_pred = classes[np.argmax(y_proba, axis=1)]

    macro_f2 = float(fbeta_score(y_test, y_pred, beta=2, average="macro"))
    macro_auc = float(roc_auc_score(y_test_bin, y_proba, multi_class="ovr", average="macro"))
    macro_ap = float(average_precision_score(y_test_bin, y_proba, average="macro"))

    results_all_vars_rfe[name] = {
        "macro_f2": macro_f2,
        "macro_auc": macro_auc,
        "macro_avg_precision": macro_ap,
        "cv_macro_f2_mean": cv_f2_mean,
        "cv_macro_f2_std": cv_f2_std,
        "perm_cv_macro_f2_mean": perm_f2_mean,
        "perm_cv_macro_f2_std": perm_f2_std,
    }
    curves_all_rfe[name]["roc"] = [arr.tolist() for arr in roc_curve(y_test_bin.ravel(), y_proba.ravel())[:2]]
    curves_all_rfe[name]["pr"] = [
        arr.tolist() for arr in precision_recall_curve(y_test_bin.ravel(), y_proba.ravel())[:2]
    ]

    best_score = -np.inf
    best_subset = []
    best_model = None
    best_metrics = {}

    for n_feats in feature_counts:
        if n_feats >= X_train_balanced.shape[1]:
            continue  # skip if request exceeds available features

        rfe = RFE(
            estimator=clone(estimator),
            n_features_to_select=n_feats,
            step=1,
        )
        rfe.fit(X_train_balanced, y_train_balanced)

        subset = X_train_balanced.columns[rfe.support_]
        subset_list = list(subset)
        model_sel = clone(estimator)
        model_sel.fit(X_train_balanced[subset_list], y_train_balanced)
        y_proba_sel = model_sel.predict_proba(X_test_scaled[subset_list])
        y_pred_sel = model_sel.classes_[np.argmax(y_proba_sel, axis=1)]
        y_test_bin_sel = label_binarize(y_test, classes=model_sel.classes_)

        macro_f2_sel = fbeta_score(y_test, y_pred_sel, beta=2, average="macro")
        macro_auc_sel = roc_auc_score(
            y_test_bin_sel,
            y_proba_sel,
            multi_class="ovr",
            average="macro",
        )
        macro_ap_sel = average_precision_score(y_test_bin_sel, y_proba_sel, average="macro")

        if macro_f2_sel > best_score:
            best_score = macro_f2_sel
            best_subset = subset_list
            best_model = model_sel
            best_metrics = {
                "macro_f2": float(macro_f2_sel),
                "macro_auc": float(macro_auc_sel),
                "macro_avg_precision": float(macro_ap_sel),
                "roc": [
                    arr.tolist() for arr in roc_curve(y_test_bin_sel.ravel(), y_proba_sel.ravel())[:2]
                ],
                "pr": [
                    arr.tolist()
                    for arr in precision_recall_curve(y_test_bin_sel.ravel(), y_proba_sel.ravel())[:2]
                ],
            }

        print(
            f"  n_features={n_feats:<3d} | F2-macro={macro_f2_sel:.3f} | AUC={macro_auc_sel:.3f} | AP={macro_ap_sel:.3f}"
        )

    important_features_rfe[name] = list(best_subset)
    results_rfe[name] = {
        "macro_f2": best_metrics["macro_f2"],
        "macro_auc": best_metrics["macro_auc"],
        "macro_avg_precision": best_metrics["macro_avg_precision"],
        "selected_features": list(best_subset),
    }
    curves_rfe[name]["roc"] = best_metrics["roc"]
    curves_rfe[name]["pr"] = best_metrics["pr"]

    print("Best subset for", name)
    print(classification_report(y_test, best_model.predict(X_test_scaled[list(best_subset)])))
    print(
        f"Selected {len(best_subset)} features | F2-macro={best_metrics['macro_f2']:.3f} | AUC={best_metrics['macro_auc']:.3f}"
    )


Modelo: Random Forest
  CV stratified F2-macro=0.236 ± 0.012 (training-only validation)
  Permuted-label CV F2-macro=0.124 ± 0.004 (sanity check)
  n_features=40  | F2-macro=0.229 | AUC=0.732 | AP=0.249
  n_features=20  | F2-macro=0.205 | AUC=0.715 | AP=0.241
  n_features=15  | F2-macro=0.211 | AUC=0.692 | AP=0.226
  n_features=10  | F2-macro=0.212 | AUC=0.687 | AP=0.223
Best subset for Random Forest
              precision    recall  f1-score   support

           0       0.43      0.49      0.46       251
           1       0.26      0.18      0.22        60
           2       0.68      0.77      0.72       597
           3       0.00      0.00      0.00        22
           4       0.18      0.08      0.11        39
           5       0.50      0.03      0.06        29
           6       0.12      0.04      0.06        51

    accuracy                           0.57      1049
   macro avg       0.31      0.23      0.23      1049
weighted avg       0.53      0.57      0.54      1049


In [ ]:
previous_training = {
    "curves2_rfeadj": curves2_rfe,
    "results2_rfeadj": results2_rfe,
    "results2_all_vars_rfeadj": results2_all_vars_rfe,
    "important_features2_rfeadj": important_features2_rfe,
    "curves_rfe": curves_rfe,
    "results_rfe": results_rfe,
    "results_all_vars_rfe": results_all_vars_rfe,
    "important_features_rfe": important_features_rfe
}

In [12]:
for key, values in results_rfe.items():
    print(f"\nModel: {key}")
    print(f"  F2-macro: {values['macro_f2']:.3f}")
    print(f"  AUC-macro: {values['macro_auc']:.3f}")
    print(f"  AP-macro: {values['macro_avg_precision']:.3f}")
    print(f"  Selected ({len(values['selected_features'])} features ): {list(values['selected_features'])}")


Model: Random Forest
  F2-macro: 0.178
  AUC-macro: 0.713
  AP-macro: 0.284
  Selected (10 features ): ['edad', 'indice_de_charlson', 'foco', 'respiracion', 'plaquetas', 'creatinina', 'temperatura', 'frec_cardiaca', 'tension_arterial', 'tension_arterial_recoded']

Model: Logistic Regression
  F2-macro: 0.239
  AUC-macro: 0.692
  AP-macro: 0.274
  Selected (40 features ): ['edad', 'paciente_residencia', 'colagenopatia', 'hemiplejia', 'linfoma', 'leucemia', 'indice_de_charlson', 'inmunosupresion', 'cirugia_previa_sin_implant', 'hemodialisis_permanente', 'cateter_venoso', 'sonda_urinaria', 'valvula_prot_cardiaca', 'portador_otros_disposit', 'foco', 'cardiovascular', 'plaquetas', 'frec_cardiaca', 'hipoxemia', 'sintoma_dolor_abdominal', 'sintoma_lesión_de_la_piel', 'sintoma_lesión_de_mucosa', 'sintoma_cefalea', 'sintoma_dolor_articular', 'sintoma_dificultad_para_respirar', 'sintoma_disuria', 'sintoma_síndrome_de_disuria_-_polaquiuria', 'sintoma_dolor_en_el_ángulo_renal', 'sintoma_náuseas',

## RFECV approach

Apply cross-validated recursive elimination to determine feature counts that generalise best.

### RFECV implementation details

Describe cross-validation splits, scoring metrics, and stopping criteria for RFECV runs.

In [50]:
results_all_vars_rfecv, results_rfecv = {}, {}
curves_all_rfecv, curves_rfecv = defaultdict(dict), defaultdict(dict)
important_features_rfecv = {}
for name, estimator in models.items():
    print(f"Modelo: {name}")
    model = clone(estimator)
    model.fit(X_train_balanced, y_train_balanced)

    classes = model.classes_
    y_test_bin = label_binarize(y_test, classes=classes)
    y_proba = model.predict_proba(X_test_scaled)
    y_pred = classes[np.argmax(y_proba, axis=1)]

    macro_f2 = fbeta_score(y_test, y_pred, beta=2, average="macro")
    macro_auc = roc_auc_score(y_test_bin, y_proba, multi_class="ovr", average="macro")
    macro_ap = average_precision_score(y_test_bin, y_proba, average="macro")

    fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
    prec_micro, rec_micro, _ = precision_recall_curve(y_test_bin.ravel(), y_proba.ravel())

    results_all_vars_rfecv[name] = {
        "macro_f2": macro_f2,
        "macro_auc": macro_auc,
        "macro_avg_precision": macro_ap,
    }
    curves_all_rfecv[name]["roc"] = (fpr_micro, tpr_micro)
    curves_all_rfecv[name]["pr"] = (rec_micro, prec_micro)

    rfe = RFECV(
        estimator=clone(estimator),
        step=1,
        cv=cv_kfold,
        scoring=scorer,
        n_jobs=-1,
    )
    rfe.fit(X_train_balanced, y_train_balanced)

    selected_features = X_train.columns[rfe.support_]
    important_features_rfecv[name] = selected_features

    model_sel = clone(estimator)
    model_sel.fit(X_train_balanced[selected_features], y_train_balanced)

    y_proba_sel = model_sel.predict_proba(X_test_scaled[selected_features])
    y_pred_sel = model_sel.classes_[np.argmax(y_proba_sel, axis=1)]
    y_test_bin_sel = label_binarize(y_test, classes=model_sel.classes_)

    print(classification_report(y_test, y_pred_sel))
    print(f"F2-macro: {macro_f2:.3f} | AUC-macro: {macro_auc:.3f} | AP-macro: {macro_ap:.3f}")

    macro_f2_sel = fbeta_score(y_test, y_pred_sel, beta=2, average="macro")
    macro_auc_sel = roc_auc_score(
        y_test_bin_sel, y_proba_sel, multi_class="ovr", average="macro"
    )
    macro_ap_sel = average_precision_score(y_test_bin_sel, y_proba_sel, average="macro")

    fpr_micro_sel, tpr_micro_sel, _ = roc_curve(
        y_test_bin_sel.ravel(), y_proba_sel.ravel()
    )
    prec_micro_sel, rec_micro_sel, _ = precision_recall_curve(
        y_test_bin_sel.ravel(), y_proba_sel.ravel()
    )

    results_rfecv[name] = {
        "macro_f2": macro_f2_sel,
        "macro_auc": macro_auc_sel,
        "macro_avg_precision": macro_ap_sel,
        "selected_features": selected_features,
        "model": model_sel,
    }
    curves_rfecv[name]["roc"] = (fpr_micro_sel, tpr_micro_sel)
    curves_rfecv[name]["pr"] = (rec_micro_sel, prec_micro_sel)

    print(
        f"{name} - Selected {len(selected_features)} features | "
        f"F2-macro: {macro_f2_sel:.3f} | AUC-macro: {macro_auc_sel:.3f}"
    )

print("Overall scores for all models:")
for model, scores in results_rfecv.items():
    print(f"{model}: F2-macro = {scores['macro_f2']:.3f}, AUC-macro = {scores['macro_auc']:.3f}, AP-macro = {scores['macro_avg_precision']:.3f}")

consensus_features = set.intersection(*[set(f) for f in important_features_rfecv.values()])
print("\nSelected features shared by all models:", sorted(consensus_features))


Modelo: Random Forest
              precision    recall  f1-score   support

           0       0.52      0.77      0.62       270
           1       0.38      0.18      0.25        65
           2       0.63      0.62      0.62       395
           3       0.50      0.04      0.08        23
           4       0.36      0.14      0.20        37
           5       0.15      0.11      0.13        36
           6       0.42      0.09      0.15        56

    accuracy                           0.55       882
   macro avg       0.42      0.28      0.29       882
weighted avg       0.53      0.55      0.51       882

F2-macro: 0.282 | AUC-macro: 0.756 | AP-macro: 0.322
Random Forest - Selected 109 features | F2-macro: 0.281 | AUC-macro: 0.755
Modelo: Logistic Regression
              precision    recall  f1-score   support

           0       0.52      0.35      0.42       270
           1       0.10      0.18      0.13        65
           2       0.61      0.22      0.32       395
        

# Binary RFE for sepsis model

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, f1_score, precision_recall_curve, average_precision_score
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import copy
from itertools import combinations

invalid_cols = [TARGET_VARIABLE, "qsofa", "vasopresores", "hipotension"]

"""list_of_features = ['edad', 'indice_de_charlson', 'respiracion', 'cardiovascular',
       'plaquetas', 'creatinina', 'estado_mental_alterado', 'temperatura',
       'frec_cardiaca', 'taquipnea', 'tension_arterial', 'hipotension',
       'saturacion_o2', 'tension_arterial_recoded']
test_df = copy.deepcopy(processed_df)
test_df = create_combinations(test_df, list_of_features, drop=True)"""
test_df = copy.deepcopy(processed_df)

X = test_df.drop(columns=invalid_cols)
y = test_df[TARGET_VARIABLE].astype(int)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=99)

X_train = pd.DataFrame(X_train, columns=X.columns).reset_index(drop=True)
X_test = pd.DataFrame(X_test, columns=X.columns).reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

models = {
    "Random Forest": RandomForestClassifier(random_state=99, class_weight="balanced"),
    "SVM": SVC(kernel="linear", probability=True, random_state=99),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=99, class_weight="balanced"),
    "Xgboost Tree": XGBClassifier(random_state=99),
}

results_all_vars = {}
results = {}
important_features = {}

for name, model in models.items():
    
    print(f"\nModelo: {name}")
    if name == "SVM":
        continue
    model.fit(X_train, y_train)
    y_pred_prob_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = model.predict(X_test)

    fpr_test, tpr_test, _ = roc_curve(y_test, y_pred_prob_test)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_prob_test)
    auc_test = auc(fpr_test, tpr_test)
    avg_precision = average_precision_score(y_test, y_pred_prob_test)
    f1_test = f1_score(y_test, y_pred_test)
    
    results_all_vars[name] = {
        "fpr_test": fpr_test,
        "tpr_test": tpr_test,
        "precision": precision,
        "recall": recall,
        "auc_test": auc_test,
        "avg_precision": avg_precision,
        "f1_test": f1_test,
    }

    print(f"AUC (all features): {auc_test:.3f}, Avg Precision: {avg_precision:.3f}, F1: {f1_test:.3f}")
    
    print(f"\Training {name} with RFECV ...")

    auc_test_scores = []
    f1_test_scores = []

    rfe = RFECV(estimator=model,
                step=1,
                cv=StratifiedKFold(n_splits=5),
                scoring=make_scorer(fbeta_score, beta=2, average="macro"))
    
    rfe.fit(X_train, y_train)

    print(f"Optimum number of characteristics for {name}: {rfe.n_features_}")
    
    final_features = X_train.columns[rfe.support_]
    important_features[name] = final_features

    X_train_rfe = X_train[final_features]
    X_test_rfe = X_test[final_features]

    model.fit(X_train_rfe, y_train)

    y_pred_prob_test = model.predict_proba(X_test_rfe)[:, 1]
    y_pred_test = model.predict(X_test_rfe)

    fpr_test, tpr_test, _ = roc_curve(y_test, y_pred_prob_test)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_prob_test)
    auc_test = auc(fpr_test, tpr_test)
    avg_precision = average_precision_score(y_test, y_pred_prob_test)
    f1_test = f1_score(y_test, y_pred_test)

    results[name] = {
        "fpr_test": fpr_test,
        "tpr_test": tpr_test,
        "precision": precision,
        "recall": recall,
        "auc_test": auc_test,
        "avg_precision": avg_precision,
        "f1_test": f1_test,
        "selected_features": final_features,
    }
    results[name]["importances"] = model.feature_importances_ if hasattr(model, "feature_importances_") else []

    print(f"{name} - AUC Test: {auc_test:.3f}, F1 Test: {f1_test:.3f}")
    print(f"Características seleccionadas por {name}: {final_features}")

# ROC
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
for model_name in results_all_vars:
    metrics = results_all_vars[model_name]
    plt.plot(metrics["fpr_test"], metrics["tpr_test"], label=f"{model_name} (AUC: {metrics['auc_test']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random (AUC: 0.500)")
plt.title("Comparison of models by ROC curves (all features)")
plt.xlabel("1 - Specificity")
plt.ylabel("Sensitivity")
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
for model_name, metrics in results.items():
    plt.plot(metrics["fpr_test"], metrics["tpr_test"], label=f"{model_name} (AUC: {metrics['auc_test']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random (AUC: 0.500)")
plt.title("Comparison of models by ROC curves")
plt.xlabel("1 - Specificity")
plt.ylabel("Sensitivity")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

# RECALL
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
for model_name, metrics in results_all_vars.items():
    plt.plot(metrics["recall"], metrics["precision"], label=f"{model_name} (Avg Precision: {metrics['avg_precision']:.3f})")
plt.axhline(y=0.5, color="r", linestyle="--", label="Precisión = 0.500")
plt.title("Precision-Recall - All features")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
for model_name, metrics in results.items():
    plt.plot(metrics["recall"], metrics["precision"], label=f"{model_name} (Avg Precision: {metrics['avg_precision']:.3f})")
plt.axhline(y=0.5, color="r", linestyle="--", label="Precisión = 0.500")
plt.title("Precision-Recall - Selected features")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, len(important_features), figsize=(20, 8), sharey=True)
for ax, (model_name, features) in zip(axes, important_features.items()):
    ax.barh(features, [1 + l for l in range(len(features))])
    ax.set_title(model_name)
    ax.set_xlabel("Importance ranking")
plt.tight_layout()
plt.show()

consensus_features = set(important_features[list(important_features.keys())[0]])
for features in important_features.values():
    consensus_features &= set(features)

print("\nSelected features: ", list(consensus_features))